In [11]:
from datasets import load_dataset

# Load JSONL file for cross verifying
data_ft = load_dataset("json", data_files="/content/updated_data.jsonl", split="train")

# display the first few rows
print(data_ft[0])
print(data_ft[1])
print(data_ft[2])



{'question': "What services does Iron Mountain provide to protect organizations' information and reduce storage costs?", 'answer': 'Iron Mountain provides services such as storing physical records and data backup media, offering information management solutions, and providing data center space for enterprise-class colocation and hyperscale deployments.', 'context': 'Iron Mountain helps organizations protect their information and reduce storage costs by storing physical records and data backup media, offering information management solutions, and providing data center space.'}
{'question': 'How is inventory valued for financial reporting?', 'answer': 'Inventory is valued at standard cost adjusted to the lower of actual cost or estimated net realizable value based on future demand and market conditions.', 'context': 'We value inventory at standard cost, adjusted to approximate the lower of actual cost or estimated net realizable value using assumptions about future demand and market cond

In [12]:
# build the single training string for LLaMa fientuning
def to_text(ex):
    return {
        "text": f"Question: {ex['question']}\nContext: {ex['context']}\nAnswer: {ex['answer']}"
    }

data_ft = data_ft.map(to_text)

# keep only the text column
data_ft = data_ft.remove_columns([c for c in data_ft.column_names if c != "text"])

# quick check
print(data_ft[0]["text"])


Map:   0%|          | 0/700 [00:00<?, ? examples/s]

Question: What services does Iron Mountain provide to protect organizations' information and reduce storage costs?
Context: Iron Mountain helps organizations protect their information and reduce storage costs by storing physical records and data backup media, offering information management solutions, and providing data center space.
Answer: Iron Mountain provides services such as storing physical records and data backup media, offering information management solutions, and providing data center space for enterprise-class colocation and hyperscale deployments.


In [13]:
data_ft

Dataset({
    features: ['text'],
    num_rows: 700
})

In [15]:
## Display the first few rows of the dataset

for i in range(5):
    print(data_ft[i]["text"])
    print("-----")

Question: What services does Iron Mountain provide to protect organizations' information and reduce storage costs?
Context: Iron Mountain helps organizations protect their information and reduce storage costs by storing physical records and data backup media, offering information management solutions, and providing data center space.
Answer: Iron Mountain provides services such as storing physical records and data backup media, offering information management solutions, and providing data center space for enterprise-class colocation and hyperscale deployments.
-----
Question: How is inventory valued for financial reporting?
Context: We value inventory at standard cost, adjusted to approximate the lower of actual cost or estimated net realizable value using assumptions about future demand and market conditions.
Answer: Inventory is valued at standard cost adjusted to the lower of actual cost or estimated net realizable value based on future demand and market conditions.
-----
Question: 

In [25]:
# Convert data_ft into prompt/completion
# THIS LINES OF CODE DIRECTLY USED FROM CHATGPT

from datasets import Dataset
import re

ANS_TAG = "Answer:"

# PROMPT COMPLETION
def to_prompt_completion(ex):
    s = ex["text"].strip()
    i = s.find(ANS_TAG)
    if i == -1:
        return {"prompt": "", "completion": ""}
    prompt = s[: i + len(ANS_TAG)].strip()
    completion = s[i + len(ANS_TAG):].strip()
    if completion and not completion.startswith(" "):
        completion = " " + completion
    return {"prompt": prompt, "completion": completion}

data_pc = data_ft.map(to_prompt_completion)

# IF ANY MISING ROW EXIST DROP
data_pc = data_pc.filter(lambda ex: len(ex["prompt"].strip()) > 0 and len(ex["completion"].strip()) > 0)

# prompt+completion
def _mk_key(ex):
    return {"_k": (ex["prompt"] + "||" + ex["completion"]).lower()}

data_pc = data_pc.map(_mk_key)
_seen = set()
def _keep_unique(ex):
    k = ex["_k"]
    if k in _seen:
        return False
    _seen.add(k)
    return True

data_pc = data_pc.filter(_keep_unique).remove_columns(["_k"])

# SHUFFLE AND SPLIT THE DATSET INTO TRAIN AND VALIDATIONS SET
data_pc = data_pc.shuffle(seed=42)
split = data_pc.train_test_split(test_size=0.1, seed=42)
train_pc, val_pc = split["train"], split["test"]

print("Train rows:", train_pc.num_rows)
print("Val rows:", val_pc.num_rows)

# Display few lines

print("PROMPT PREVIEW:\n", train_pc[4]["prompt"][:300])
print("\nCOMPLETION PREVIEW:\n", train_pc[4]["completion"][:200])



Map:   0%|          | 0/700 [00:00<?, ? examples/s]

Filter:   0%|          | 0/700 [00:00<?, ? examples/s]

Map:   0%|          | 0/700 [00:00<?, ? examples/s]

Filter:   0%|          | 0/700 [00:00<?, ? examples/s]

Train rows: 630
Val rows: 70
PROMPT PREVIEW:
 Question: What advancements characterize the AMD EPYC 9004 Series processors?
Context: Our 4th Gen AMD EPYC 9004 Series processors are built on the “Zen 4” 5 nanometer (nm) process node and are designed to deliver leadership performance and energy efficiency across a range of market segments and wor

COMPLETION PREVIEW:
  The AMD EPYC 9004 Series processors are built on the “Zen 4” 5 nanometer process node and are designed to deliver leadership performance and energy efficiency for a range of market segments and workl


In [26]:
# save the formatted dataset
data_pc.select_columns(["prompt","completion"]).to_json(
    "finance_pc_all.jsonl", orient="records", lines=True, force_ascii=False
)

train_pc.select_columns(["prompt","completion"]).to_json(
    "finance_pc_train.jsonl", orient="records", lines=True, force_ascii=False
)
val_pc.select_columns(["prompt","completion"]).to_json(
    "finance_pc_val.jsonl", orient="records", lines=True, force_ascii=False
)

print("Wrote finance_pc_all.jsonl, finance_pc_train.jsonl, finance_pc_val.jsonl")


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Wrote finance_pc_all.jsonl, finance_pc_train.jsonl, finance_pc_val.jsonl


In [27]:
from datasets import load_dataset

# Load the three datasets back
all_ds = load_dataset("json", data_files="finance_pc_all.jsonl", split="train")
train_ds = load_dataset("json", data_files="finance_pc_train.jsonl", split="train")
val_ds   = load_dataset("json", data_files="finance_pc_val.jsonl", split="train")

# inspecting all the content
def inspect_dataset(name, ds, n=2):
    print(f"=== {name.upper()} ===")
    print("Rows:", ds.num_rows)
    print("Columns:", ds.column_names)
    print("Sample rows:")
    for i in range(min(n, ds.num_rows)):
        print("Prompt:", ds[i]["prompt"][:200], "...")
        print("Completion:", ds[i]["completion"][:200], "\n")
    print("="*60)

# Inspect all three
inspect_dataset("all", all_ds)
inspect_dataset("train", train_ds)
inspect_dataset("validation", val_ds)


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

=== ALL ===
Rows: 700
Columns: ['prompt', 'completion']
Sample rows:
Prompt: Question: How are advertising costs handled in accounting according to the noted business's financial statements for the specified years?
Context: Advertising Costs Costs for advertising are expensed  ...
Completion:  Advertising costs are expensed the first time the advertising takes place or as incurred. For the years ending December 31, 2023, 2022, and 2021, the recorded advertising costs were $47 million, $29  

Prompt: Question: What percentage increase did Kroger's net earnings per diluted common share experience from 2021 to 2022?
Context: During 2022, Kroger's net earnings per diluted common share escalated to $3 ...
Completion:  41% 

=== TRAIN ===
Rows: 630
Columns: ['prompt', 'completion']
Sample rows:
Prompt: Question: In which note can further details on Legal Proceedings be found within the Consolidated Financial Statements?
Context: Item 3. Legal Proceedings, which covers litigation and regulato